In [1]:
from utils import BaseImageFolderDataset

from tqdm import tqdm

import torch

from torch.utils.data import DataLoader

from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms

In [2]:
GPU = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
BATCH_SIZE = 16

In [3]:
def get_resnet_classname(probs: torch.tensor) -> list[str]:
    categories = ResNet50_Weights.IMAGENET1K_V1.meta['categories']
    
    if probs.dim == 1:
        probs = probs.unsqueeze(0)
    
    return [categories[idx] for idx in probs.argmax(dim=1)]

In [4]:
def accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
            
        return res

In [5]:
class ImageNetSketch(BaseImageFolderDataset):
    URL = 'https://www.kaggle.com/api/v1/datasets/download/wanghaohan/imagenetsketch'
    ARCHIVE_NAME = 'ImageNet-Sketch.zip'
    EXTRACTED_FOLDER = 'imagenet-sketch/sketch'

In [6]:
class VisDA2017(BaseImageFolderDataset):
    URL = 'http://csr.bu.edu/ftp/visda17/clf/train.tar'
    ARCHIVE_NAME = 'train.tar'
    EXTRACTED_FOLDER = 'train'

In [7]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
dataset = ImageNetSketch('./data', transform=preprocess, download=True)
# dataset = VisDA2017('./data', transform=preprocess, download=True)

Progress: 100.0% (7.17 GB / 7.17 GB)                                                               

Download complete! Total: 7.17 GB
Extracting VisDA2017...
Extracted!


In [8]:
len(dataset)

50889

In [ ]:
dataloader = DataLoader(
    dataset=dataset,
    batch_size=BATCH_SIZE,
    shuffle=True, 
    num_workers=0,
    pin_memory=True,
    # prefetch_factor=2
)

In [10]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(GPU)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
acc1 = 0
acc5 = 0 

for x_batch, y_batch in tqdm(dataloader):
    x_batch = x_batch.to(GPU)
    y_batch = y_batch.to(GPU)
    
    pred = model(x_batch)
    acc = accuracy(pred, y_batch)
    
    acc1 += float(acc[0])
    acc5 += float(acc[1])
    
acc1 /= len(dataloader)
acc5 /= len(dataloader)

print(f'acc@1: {acc1:.1f}, acc@5: {acc5:.1f}')

100%|██████████| 3181/3181 [13:15<00:00,  4.00it/s]

acc@1: 24.1 | acc@5: 41.3


| Model    | acc@1 | acc@5 |
|----------|-------|-------|
| ResNet18 | 20.2  | 37.3  |
| ResNet50 | 24.1  | 41.3  |